In [ ]:
# ==========================================
# ARCHITECT AI: DYNAMIC ANALYSIS (V10.7)
# ==========================================

import subprocess
import sys
import os

# --- STEP 1: INSTALLATION ---
def install_dependencies():
    print("⚙️ Verifying dependencies... (Please wait)")
    requirements = [
        "gradio", "git+https://github.com/huggingface/transformers",
        "accelerate", "qwen-vl-utils", "networkx", "easyocr",
        "bitsandbytes>=0.43.0", "einops"
    ]
    for package in requirements:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", package, "--quiet"])
        except: pass
    print("✅ Dependencies ready.")

install_dependencies()

# --- STEP 2: IMPORTS ---
import torch
import gradio as gr
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from PIL import Image, ImageEnhance
import easyocr
import numpy as np

# --- STEP 3: AI BACKEND ---
class ArchitectBackend:
    def __init__(self):
        print("🚀 Booting AI Engine...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.reader = easyocr.Reader(['en'], gpu=(self.device == 'cuda'))

        print("   - Loading Adaptive Brain...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
            llm_int8_enable_fp32_cpu_offload=True
        )
        self.model = Qwen2VLForConditionalGeneration.from_pretrained(
            "Qwen/Qwen2-VL-7B-Instruct", quantization_config=bnb_config,
            device_map="auto", offload_folder="offload_weights"
        )
        self.processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
        self.conversation_history = []
        print("✅ Backend Ready.")

    def _preprocess_image(self, image):
        pil_image = Image.fromarray(image)
        enhancer = ImageEnhance.Contrast(pil_image)
        pil_image = enhancer.enhance(2.0)
        enhancer = ImageEnhance.Sharpness(pil_image)
        return enhancer.enhance(2.0)

    def _translate_text(self, text, target_language):
        self.conversation_history = [
            {
                "role": "system",
                "content": [{"type": "text", "text": f"You are a highly efficient Expert Technical Translator for {target_language}."}]
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": (
                    f"Translate this technical text into {target_language} clearly and professionally.\n\n"
                    f"RULES:\n"
                    f"1. Preserve ALL Markdown formatting perfectly.\n"
                    f"2. Keep universal IT jargon (e.g., API, Gateway, Server, Database) in English.\n"
                    f"3. Do NOT add extra conversational commentary.\n\n"
                    f"TEXT TO TRANSLATE:\n{text}"
                )}]
            }
        ]
        return self._generate(max_tokens=1024, temperature=0.1, repetition_penalty=1.0)

    def summarize(self, image, language="English"):
        if image is None:
            return "Please upload an image first."

        self.current_image = self._preprocess_image(image)

        self.conversation_history = [
            {"role": "system", "content": [{"type": "text", "text": "You are a direct and concise Technical Expert."}]}
        ]

        prompt = "Analyze this technical diagram and provide a highly concise 2-sentence overview of its core purpose in English. No fluff."

        self.conversation_history.append({
            "role": "user",
            "content": [{"type": "image", "image": self.current_image}, {"type": "text", "text": prompt}],
        })

        english_summary = self._generate(max_tokens=256, temperature=0.2, repetition_penalty=1.05)

        if language.lower() == "english":
            return english_summary

        return self._translate_text(english_summary, language)

    def analyze(self, image, language="English"):
        if image is None:
            return "Please upload an image first."

        self.current_image = self._preprocess_image(image)

        self.conversation_history = [
            {"role": "system", "content": [{"type": "text", "text": "You are an Expert Technical Architect. You must analyze the visual flow and connections in diagrams, not just read the text."}]}
        ]

        try:
            img_np = np.array(self.current_image)
            ocr_results = self.reader.readtext(img_np, detail=0)
            ocr_hints = ", ".join(ocr_results[:60])
        except:
            ocr_hints = "No text detected."

        english_rulebook = (
            "--- RULEBOOK ---\n"
            "STEP 1: Identify the Diagram Type (System Architecture, Sequence Diagram, Flowchart, or ER Diagram).\n"
            "STEP 2: Explain it using the exact structure below based on the type you identified:\n\n"

            "IF ARCHITECTURE/CLOUD:\n"
            "1. **Diagram Type**: System Architecture\n"
            "2. **Goal**: 1-2 sentence core purpose.\n"
            "3. **Domains**: Bullet points of layers/components.\n"
            "4. **Data Flow**: Brief step-by-step path.\n\n"

            "IF SEQUENCE DIAGRAM:\n"
            "1. **Diagram Type**: Sequence Diagram\n"
            "2. **Participants**: List the distinct Actors/Systems found at the top of the vertical lifelines.\n"
            "3. **Timeline**: You MUST trace the visual arrows from top to bottom. For EVERY arrow, explicitly state who is sending it and who is receiving it using EXACTLY this format:\n"
            "   - Step #: [Sender Name] -> [Receiver Name]: [Message or Action Text]\n\n"

            "IF ER DIAGRAM / DATABASE:\n"
            "1. **Diagram Type**: Entity-Relationship (ERD)\n"
            "2. **Core Entities**: List the main data tables.\n"
            "3. **Relationships**: Describe how they connect.\n\n"

            "IF FLOWCHART/PROCESS:\n"
            "1. **Diagram Type**: Process Flow\n"
            "2. **Start/End**: Triggers and Terminations.\n"
            "3. **Steps**: The linear path and decision points."
        )

        prompt = (
            f"Analyze this diagram in English. Focus intensely on where arrows start and where they end.\n\n"
            f"The following text labels were extracted to help you identify components: [{ocr_hints}]\n\n"
            f"CRITICAL INSTRUCTION: Do NOT simply read the text labels left to right. You MUST interpret the visual connections.\n\n"
            f"{english_rulebook}\n"
        )

        self.conversation_history.append({
            "role": "user",
            "content": [{"type": "image", "image": self.current_image}, {"type": "text", "text": prompt}],
        })

        english_analysis = self._generate(max_tokens=1024, temperature=0.3, repetition_penalty=1.05)

        if language.lower() == "english":
            return english_analysis

        return self._translate_text(english_analysis, language)

    def chat_response(self, message, history, language, image=None):
        try:
            if image is not None:
                self.current_image = self._preprocess_image(image)

            if language.lower() != "english":
                 self.conversation_history = [
                    {"role": "system", "content": [{"type": "text", "text": "You are a direct translator. Translate to English."}]},
                    {"role": "user", "content": [{"type": "text", "text": message}]}
                 ]
                 english_message = self._generate(max_tokens=256, temperature=0.1, repetition_penalty=1.0)
            else:
                 english_message = message

            self.conversation_history = [
                {"role": "system", "content": [{"type": "text", "text": "You are an Expert Technical Architect."}]}
            ]

            for msg in history:
                self.conversation_history.append({
                    "role": msg["role"],
                    "content": [{"type": "text", "text": msg["content"]}]
                })

            current_turn = []
            if hasattr(self, 'current_image') and self.current_image is not None:
                current_turn.append({"type": "image", "image": self.current_image})

            current_turn.append({"type": "text", "text": english_message})
            self.conversation_history.append({"role": "user", "content": current_turn})

            english_response = self._generate(max_tokens=512, temperature=0.2, repetition_penalty=1.05)

            if language.lower() == "english":
                response = english_response
            else:
                response = self._translate_text(english_response, language)

            return response

        except Exception as e:
            return f"⚠️ System Error: Something went wrong processing that request. Details: {str(e)}"

    def _generate(self, max_tokens, temperature=0.1, repetition_penalty=1.0):
        text = self.processor.apply_chat_template(
            self.conversation_history, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(self.conversation_history)

        if image_inputs is None:
             inputs = self.processor(text=[text], padding=True, return_tensors="pt").to(self.model.device)
        else:
             inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            repetition_penalty=repetition_penalty,
            temperature=temperature
        )
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        return self.processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

# Init Backend
if 'backend' not in globals():
    backend = ArchitectBackend()

# --- STEP 4: INTERFACE ---
# BUG FIX: Removed theme="soft" from here to comply with Gradio 6.0 specs
with gr.Blocks(title="Technical Diagram Analyzer") as demo:
    gr.Markdown("# 🧠 Technical Diagram Analyzer")
    gr.Markdown("Upload ANY technical diagram. Get deep, type-aware analysis in your preferred language.")

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(label="Upload Diagram", type="numpy")

            language_dropdown = gr.Dropdown(
                choices=["English", "Hindi", "Spanish", "Chinese", "Japanese", "French", "German"],
                value="English",
                label="Output Language",
                interactive=True
            )

            with gr.Row():
                analyze_btn = gr.Button("🔍 Deep Analyze", variant="primary")
                summarize_btn = gr.Button("📝 Quick Summary", variant="secondary")

            report_output = gr.Markdown(label="Intelligent Analysis")

        with gr.Column(scale=1):
            # BUG FIX: Removed type="messages", as it's now the default in Gradio 6.0 and explicit declaration throws an error
            chatbot = gr.Chatbot(label="💬 Multi-Lingual Q&A", height=500)

            with gr.Row():
                chat_input = gr.Textbox(show_label=False, placeholder="Ask a follow-up question...", scale=4)
                chat_submit = gr.Button("Send", variant="primary", scale=1)

            def process_chat(message, history, language, image):
                history = history or []
                if not message.strip():
                    return "", history

                bot_response = backend.chat_response(message, history, language, image)

                history.append({"role": "user", "content": message})
                history.append({"role": "assistant", "content": bot_response})

                return "", history

            chat_submit.click(fn=process_chat, inputs=[chat_input, chatbot, language_dropdown, image_input], outputs=[chat_input, chatbot])
            chat_input.submit(fn=process_chat, inputs=[chat_input, chatbot, language_dropdown, image_input], outputs=[chat_input, chatbot])

    analyze_btn.click(fn=backend.analyze, inputs=[image_input, language_dropdown], outputs=[report_output])
    summarize_btn.click(fn=backend.summarize, inputs=[image_input, language_dropdown], outputs=[report_output])

# --- STEP 5: LAUNCH ---
print("🚀 Launching Interface...")
# BUG FIX: Moved theme="soft" to the launch method
demo.launch(inline=False, share=True, debug=False, prevent_thread_lock=True, theme="soft")